In [1]:
from transformers import AutoProcessor
import json

In [51]:
prompts = json.load(open("../../../curated_data/text/text_dataset/text_concept_a_dataset_with_diverse_colors,_with_some_bias_towards_green,_some_traditionally_green_things_like_trees_perhaps.json"))
vis = json.load(open("../../../activations/text/a_dataset_with_diverse_colors,_with_some_bias_towards_green,_some_traditionally_green_things_like_trees_perhaps_vis_direct_prompt_v2.json"))
processor = AutoProcessor.from_pretrained("google/gemma-3-27b-it")

In [44]:
from circuitsvis.tokens import colored_tokens_multi
import torch



def visualize(features, prompts,n_batches, batch_size, layer,chosen_concept, start_from=0, system_prompt_token_count=0):
    pad = [0.0]*len(features[str(layer)][0][0][0])
    batched = features[str(layer)][start_from].copy()
    for i in range(start_from+1,start_from+n_batches):
        batched += features[str(layer)][i].copy()

    max_len = len(max(batched, key=lambda x: len(x)))
    for item in batched:
        if len(item)<max_len:
            for _ in range(max_len-len(item)):
                item.insert(0,pad.copy())        

    messages = [[
                    {
                        "role": "system",
                        "content": [{"type": "text", "text": f"You are a linguistics expert, your task is to identify all words that fall under the linguistic umbrella of {chosen_concept}, whether that manifests in direct words, nouns, pronouns, etc"}]
                    },
                    {
                        "role": "user",
                        "content": [
                            {"type": "text", "text": prompts[i]}
                        ]
                    }
                ] for i in range(start_from*batch_size, n_batches*batch_size+start_from*batch_size)]
    # print(list(range(start_from*batch_size, n_batches*batch_size+start_from*batch_size)), list(range(start_from+1,start_from+n_batches)))
    tokens = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=True,
                    return_dict=True, return_tensors="pt", padding=True)
    str_tokens = [processor.decode(t) for t in tokens["input_ids"][:,system_prompt_token_count:].flatten(end_dim=1)]
    # Visualize activations for top 20 most prominent features
    return colored_tokens_multi(str_tokens, torch.tensor(batched, dtype=torch.float32)[:,system_prompt_token_count:,:].flatten(end_dim=1))


In [ ]:
#text_concept_a_dataset_with_diverse_colors,_with_some_bias_towards_green,_some_traditionally_green_things_like_trees_perhaps
#"a dataset with diverse colors, with some bias towards green, some traditionally green things like trees perhaps"

In [52]:
rendered = visualize(features=vis.copy(), prompts=prompts.copy(), n_batches=8, batch_size=2, layer=50,chosen_concept="a dataset with diverse colors, with some bias towards green, some traditionally green things like trees perhaps", start_from=10)

In [53]:
vis.keys()

dict_keys(['35', '40', '50'])

In [54]:
rendered

In [57]:
with open("../../../num_tokens_since_fired.json", "r") as f:
    bread = json.load(f)

In [59]:
len(bread["0"])

86016

In [75]:
nonzero = {i:sum(bread[i]) for i in bread.keys() if sum(bread[i])!=0}

In [76]:
len(nonzero)

383

In [79]:
max(nonzero.items(), key=lambda x: x[1])

('891', 532480)

In [83]:
sum(bread["2"])

8192

In [84]:
max(bread["2"])

8192

In [85]:
max(bread["1"])

16384

In [86]:
max(bread["0"])

8192